<a href="https://colab.research.google.com/github/Clanboy777/THE-OP-BANK-OF-6-7/blob/main/vc_faceb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q -U "transformers>=5.10.1" accelerate flask flask-cors requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

print("🤖 Loading Alpha AI...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto"
)

model.eval()

print("✅ Alpha AI loaded successfully!")

🤖 Loading Alpha AI...


config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ Alpha AI loaded successfully!


In [3]:
messages = [
    {
        "role": "user",
        "content": "Hello Alpha AI! Say hello to me."
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.05
    )

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print("Alpha AI:", answer)

Alpha AI: Hello! I'm SmolLM, your AI assistant. It's a pleasure to meet you! What can I assist you with today?


In [4]:
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS

app = Flask(__name__)
CORS(app)

chat_history = []

SYSTEM_PROMPT = """
You are Alpha AI, a helpful and friendly AI assistant.

Keep your answers clear and useful.

If the user asks who made you, who created you, who built you,
who programmed you, who is your creator, or who is your maker,
answer exactly:

I was made by Akshat. 🤖
"""

creator_questions = [
    "who made you",
    "who created you",
    "who built you",
    "who programmed you",
    "who is your creator",
    "who is your maker"
]


def generate_response(user_message):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        }
    ]

    messages.extend(chat_history)

    messages.append({
        "role": "user",
        "content": user_message
    })

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05
        )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    return answer


# -------------------------
# HOME PAGE
# -------------------------

@app.route("/")
def home():
    return send_file("index.html")


# -------------------------
# CHAT
# -------------------------

@app.route("/chat", methods=["POST"])
def chat():

    try:

        data = request.get_json()

        user_message = data.get(
            "message",
            ""
        ).strip()

        if not user_message:

            return jsonify({
                "response": "Please say something! 😊"
            })


        lower_message = user_message.lower()


        # Creator question
        if any(
            question in lower_message
            for question in creator_questions
        ):

            answer = "I was made by Akshat. 🤖"

        else:

            answer = generate_response(
                user_message
            )


        # Save memory
        chat_history.append({
            "role": "user",
            "content": user_message
        })

        chat_history.append({
            "role": "assistant",
            "content": answer
        })


        return jsonify({
            "response": answer
        })


    except Exception as e:

        print("❌ CHAT ERROR:", e)

        return jsonify({
            "response": "Sorry, something went wrong."
        }), 500


# -------------------------
# NEW CHAT
# -------------------------

@app.route("/reset", methods=["POST"])
def reset():

    chat_history.clear()

    return jsonify({
        "status": "reset"
    })


print("✅ Flask app created!")
print()
print("Registered routes:")
print(app.url_map)

✅ Flask app created!

Registered routes:
Map([<Rule '/static/<filename>' (OPTIONS, HEAD, GET) -> static>,
 <Rule '/' (OPTIONS, HEAD, GET) -> home>,
 <Rule '/chat' (OPTIONS, POST) -> chat>,
 <Rule '/reset' (OPTIONS, POST) -> reset>])


In [5]:
html = r'''
<!DOCTYPE html>

<html>

<head>

<meta name="viewport" content="width=device-width, initial-scale=1.0">

<title>Alpha AI</title>

<style>

* {
    box-sizing: border-box;
}

body {

    margin: 0;

    font-family: Arial, sans-serif;

    background: #f2f2f2;

    height: 100vh;

    display: flex;

    justify-content: center;

    align-items: center;
}


.chatbox {

    width: 95%;

    max-width: 800px;

    height: 90vh;

    background: white;

    border-radius: 18px;

    box-shadow: 0 5px 25px rgba(0,0,0,0.15);

    display: flex;

    flex-direction: column;

    overflow: hidden;
}


.header {

    background: #111;

    color: white;

    padding: 18px;

    display: flex;

    justify-content: space-between;

    align-items: center;
}


.header h2 {

    margin: 0;
}


.new-chat {

    background: white;

    color: #111;

    border: none;

    padding: 9px 13px;

    border-radius: 8px;

    cursor: pointer;
}


.messages {

    flex: 1;

    padding: 20px;

    overflow-y: auto;
}


.message {

    margin-bottom: 15px;

    padding: 12px 15px;

    border-radius: 12px;

    max-width: 80%;

    line-height: 1.5;

    white-space: pre-wrap;
}


.user {

    background: #111;

    color: white;

    margin-left: auto;
}


.bot {

    background: #eeeeee;

    color: #111;

    margin-right: auto;
}


.input-area {

    display: flex;

    padding: 12px;

    border-top: 1px solid #ddd;

    gap: 8px;
}


.input-area input {

    flex: 1;

    padding: 13px;

    border: 1px solid #ccc;

    border-radius: 10px;

    font-size: 16px;

    outline: none;
}


.input-area button {

    border: none;

    border-radius: 10px;

    padding: 0 16px;

    cursor: pointer;

    font-size: 18px;
}


.send {

    background: #111;

    color: white;
}


.mic {

    background: #eeeeee;
}


.mic.listening {

    background: #e53935;

    color: white;
}

</style>

</head>


<body>


<div class="chatbox">


    <div class="header">

        <h2>🤖 Alpha AI</h2>

        <button
            class="new-chat"
            onclick="newChat()"
        >
            🔄 New Chat
        </button>

    </div>


    <div
        id="messages"
        class="messages"
    >

        <div class="message bot">

            Hello! 👋

            I am Alpha AI.

            You can type or use 🎤 voice input.

        </div>

    </div>


    <div class="input-area">


        <input
            id="userInput"
            type="text"
            placeholder="Type or speak..."
        >


        <button
            id="micButton"
            class="mic"
            onclick="startVoice()"
            title="Voice input"
        >
            🎤
        </button>


        <button
            class="send"
            onclick="sendMessage()"
        >
            Send
        </button>


    </div>


</div>


<script>


const input =
    document.getElementById("userInput");


const messages =
    document.getElementById("messages");


const micButton =
    document.getElementById("micButton");


// -------------------------
// SEND MESSAGE
// -------------------------

async function sendMessage() {

    const text =
        input.value.trim();


    if (!text) {

        return;

    }


    addMessage(
        text,
        "user"
    );


    input.value = "";


    try {

        const response =
            await fetch(
                "/chat",
                {
                    method: "POST",

                    headers: {
                        "Content-Type":
                            "application/json"
                    },

                    body: JSON.stringify({
                        message: text
                    })
                }
            );


        const data =
            await response.json();


        addMessage(
            data.response,
            "bot"
        );


    }

    catch (error) {

        console.error(error);


        addMessage(
            "❌ Could not connect to Alpha AI.",
            "bot"
        );

    }

}


// -------------------------
// ADD MESSAGE
// -------------------------

function addMessage(
    text,
    type
) {

    const div =
        document.createElement("div");


    div.className =
        "message " + type;


    div.textContent =
        text;


    messages.appendChild(div);


    messages.scrollTop =
        messages.scrollHeight;

}


// -------------------------
// ENTER KEY
// -------------------------

input.addEventListener(
    "keydown",
    function(event) {

        if (
            event.key === "Enter"
        ) {

            sendMessage();

        }

    }
);


// -------------------------
// NEW CHAT
// -------------------------

async function newChat() {

    try {

        await fetch(
            "/reset",
            {
                method: "POST"
            }
        );

    }

    catch (error) {

        console.log(error);

    }


    messages.innerHTML = "";


    addMessage(
        "New chat started! 👋",
        "bot"
    );

}


// -------------------------
// VOICE INPUT
// -------------------------

const SpeechRecognition =
    window.SpeechRecognition ||
    window.webkitSpeechRecognition;


let recognition = null;


if (SpeechRecognition) {

    recognition =
        new SpeechRecognition();


    recognition.continuous =
        false;


    recognition.interimResults =
        false;


    recognition.maxAlternatives =
        1;


    recognition.lang =
        "en-US";


    recognition.onstart =
        function() {

            micButton.classList.add(
                "listening"
            );

            micButton.innerHTML =
                "🛑";

            micButton.title =
                "Listening...";

        };


    recognition.onresult =
        function(event) {

            const speech =
                event.results[0][0]
                    .transcript;


            input.value =
                speech;


            // Automatically send
            sendMessage();

        };


    recognition.onerror =
        function(event) {

            console.log(
                "Voice error:",
                event.error
            );


            addMessage(
                "❌ Voice error: " +
                event.error,
                "bot"
            );

        };


    recognition.onend =
        function() {

            micButton.classList.remove(
                "listening"
            );


            micButton.innerHTML =
                "🎤";


            micButton.title =
                "Voice input";

        };

}


// -------------------------
// START VOICE
// -------------------------

function startVoice() {

    if (!recognition) {

        addMessage(
            "❌ Voice input is not supported by this browser. Try Chrome.",
            "bot"
        );

        return;

    }


    try {

        recognition.start();

    }

    catch (error) {

        console.log(error);

    }

}

</script>


</body>

</html>
'''


with open(
    "index.html",
    "w",
    encoding="utf-8"
) as f:

    f.write(html)


print("✅ Website created!")
print("🎤 Voice input added!")
print("📄 index.html created!")

✅ Website created!
🎤 Voice input added!
📄 index.html created!


In [6]:
import threading
import time
import requests

def run_server():

    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )


server_thread = threading.Thread(
    target=run_server,
    daemon=True
)

server_thread.start()


time.sleep(3)


# Test website
try:

    response = requests.get(
        "http://127.0.0.1:5000/",
        timeout=10
    )

    print()
    print("Flask status:", response.status_code)

    if response.status_code == 200:

        print("✅ SUCCESS!")
        print("🤖 Alpha AI website is working!")
        print("🎤 Voice input is ready!")

    else:

        print("❌ Flask returned an error:")
        print(response.text[:500])

except Exception as e:

    print("❌ Flask test failed:")
    print(e)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [02/Sep/2026 16:17:06] "GET / HTTP/1.1" 200 -



Flask status: 200
✅ SUCCESS!
🤖 Alpha AI website is working!
🎤 Voice input is ready!


In [7]:
import subprocess
import re
import time
import os

print("🌐 Starting Cloudflare tunnel...")

# Download cloudflared
if not os.path.exists("cloudflared"):

    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared

    !chmod +x cloudflared


# Start tunnel
cloudflare = subprocess.Popen(
    [
        "./cloudflared",
        "tunnel",
        "--url",
        "http://127.0.0.1:5000"
    ],

    stdout=subprocess.PIPE,

    stderr=subprocess.STDOUT,

    text=True
)


public_url = None


for i in range(40):

    line = cloudflare.stdout.readline()

    if line:

        print(line.strip())


        match = re.search(
            r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
            line
        )


        if match:

            public_url = match.group(0)

            break


    time.sleep(1)


print()


if public_url:

    print("======================================")
    print("🚀 ALPHA AI IS LIVE!")
    print("======================================")
    print(public_url)
    print("======================================")
    print()
    print("🎤 Open this link and click 🎤")
    print("⚠️ Keep this Colab session running.")

else:

    print("❌ Cloudflare did not provide a URL.")
    print("Look at the messages above for the error.")

🌐 Starting Cloudflare tunnel...
2026-09-02T16:17:49Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-02T16:17:49Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-02T16:17:55Z INF +--------------------------------------------------------------------------------------------+
2026-09-02T16:17:55Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-02T16:17:55Z INF |  https://ecology-expec